# What a Data Scientist Actually Does

Notebook for **Chapter 1** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch01.html

Tip: change a number, predict the output, then run the cell again.

## 1.1 The Real Job and the Role Types

## 1.2 The Project Lifecycle (CRISP-DM and Modern Variants)

## 1.3 Problem Framing Patterns

In [ ]:
import numpy as np
import pandas as pd

PRED_DATE = pd.Timestamp("2025-01-01")   # the day we "make the prediction"
END_DATE = pd.Timestamp("2025-04-30")    # last day of data we have

def make_data(n=6000, seed=42):
    """Synthetic subscription shop. Returns (customers, orders)."""
    rng = np.random.default_rng(seed)
    cid = np.arange(1, n + 1)
    signup = pd.Timestamp("2023-01-01") + pd.to_timedelta(rng.integers(0, 730, n), unit="D")
    plan = rng.choice(["free", "pro"], n, p=[0.7, 0.3])
    channel = rng.choice(["organic", "ads", "referral"], n, p=[0.5, 0.35, 0.15])
    city = rng.choice(["Mumbai", "Delhi", "Bengaluru", "Pune", "Chennai", "Other"],
                      n, p=[0.2, 0.2, 0.2, 0.1, 0.1, 0.2])
    age = np.clip(rng.normal(32, 9, n), 18, 70).round()
    rate = rng.gamma(2.0, 0.04, n) * np.where(plan == "pro", 1.6, 1.0)  # orders per day
    # churn hazard per day (higher for ads channel, free plan, low order rate)
    hazard = 0.004 * np.exp(0.5 * (channel == "ads") - 0.7 * (plan == "pro")
                            - 0.3 * (channel == "referral") - 6.0 * (rate - 0.08))
    churn_day = signup + pd.to_timedelta(rng.exponential(1 / hazard).round(), unit="D")
    rows = []
    for i in range(n):
        stop = min(churn_day[i], END_DATE)
        days = (stop - signup[i]).days
        if days <= 0:
            continue
        k = rng.poisson(rate[i] * days)
        if k == 0:
            continue
        offs = np.sort(rng.integers(0, days + 1, k))
        amt = rng.lognormal(6.0 + 0.3 * (plan[i] == "pro"), 0.6, k).round(2)
        cat = rng.choice(["grocery", "fashion", "electronics", "home"], k, p=[0.5, 0.2, 0.1, 0.2])
        for o, a, c in zip(offs, amt, cat):
            rows.append((cid[i], signup[i] + pd.Timedelta(days=int(o)), a, c))
    orders = pd.DataFrame(rows, columns=["customer_id", "order_ts", "amount", "category"])
    orders.insert(0, "order_id", np.arange(1, len(orders) + 1))
    customers = pd.DataFrame({"customer_id": cid, "signup_date": signup, "plan": plan,
                              "channel": channel, "city": city, "age": age})
    return customers, orders

def make_label_table(customers, orders, pred_date=PRED_DATE, active_days=90, horizon=60):
    """One row per ACTIVE customer at pred_date; label = no order in next `horizon` days."""
    past = orders[orders.order_ts < pred_date]
    last = past.groupby("customer_id").order_ts.max()
    active = last[last >= pred_date - pd.Timedelta(days=active_days)].index
    fut = orders[(orders.order_ts >= pred_date) &
                 (orders.order_ts < pred_date + pd.Timedelta(days=horizon))]
    buyers = set(fut.customer_id)
    df = customers[customers.customer_id.isin(active) & (customers.signup_date < pred_date)].copy()
    df["churn"] = (~df.customer_id.isin(buyers)).astype(int)
    return df.reset_index(drop=True)

In [ ]:
from ds_common import *
customers, orders = make_data()          # seed 42, 6000 customers
d = make_label_table(customers, orders)  # active customers at 2025-01-01
print(customers.shape, orders.shape)
print(d.shape, round(d.churn.mean(), 3))

(6000, 6) (151719, 5)
(2562, 7) 0.254


## 1.4 Baselines, Success Metrics and the First Model

In [ ]:
from ds_common import *
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score

c, o = make_data()
d = make_label_table(c, o)
past = o[o.order_ts < PRED_DATE]                 # only data before T
g = past.groupby("customer_id")
f = pd.DataFrame({
    "recency_days": (PRED_DATE - g.order_ts.max()).dt.days,
    "orders_90d": past[past.order_ts >= PRED_DATE - pd.Timedelta(days=90)]
                      .groupby("customer_id").size(),
})
d = d.merge(f, left_on="customer_id", right_index=True, how="left")
d["is_pro"] = (d.plan == "pro").astype(int)

tr, te = train_test_split(d, test_size=0.3, random_state=0, stratify=d.churn)
print("B0 accuracy (predict nobody churns):", round(1 - te.churn.mean(), 3))
rule = (te.recency_days > 30).astype(int)
print("B1 rule recency>30: precision", round(precision_score(te.churn, rule), 3),
      "recall", round(recall_score(te.churn, rule), 3),
      "AUC", round(roc_auc_score(te.churn, te.recency_days), 3))
X = ["recency_days", "orders_90d", "is_pro"]
m = LogisticRegression(max_iter=1000).fit(tr[X], tr.churn)
p = m.predict_proba(te[X])[:, 1]
print("M1 logistic AUC", round(roc_auc_score(te.churn, p), 3))
k = int(0.2 * len(te))
top = te.assign(p=p).sort_values("p", ascending=False).head(k)
print("top 20% size", k, "churn rate inside", round(top.churn.mean(), 3),
      "lift", round(top.churn.mean() / te.churn.mean(), 2))
top_r = te.sort_values("recency_days", ascending=False).head(k)
print("rule-based top 20% churn rate", round(top_r.churn.mean(), 3))

## Chapter recap

## Practice set

Problem 1.P.1 (easy) A project lasts 15 weeks. Using 60/20/20, how many weeks go to modelling? **Answer:** 20% of 15 = 3 weeks.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch01.html